# Notebook 5 - Feature Engineering and Fitted Preprocessing

## Purpose

Turn the training data and Notebook 4 findings into a reproducible,
model-ready feature table.

This notebook performs **feature engineering and preprocessing only**.

It does **not** train or evaluate a machine-learning model.

## Input artifacts

- `artifacts/notebook4_eda_findings.json`
- `data/processed/train.parquet`

## Output artifacts

- `artifacts/notebook5_train_features.parquet`
- `artifacts/notebook5_preprocessor.joblib`
- `artifacts/notebook5_feature_list.json`
- `artifacts/notebook5_manifest.json`

## Important fitting rule

All fitted preprocessing objects are learned from the **training split
only**.

Validation and test remain unopened.

Notebook 6 must load and reuse the fitted preprocessing object.
It must never fit preprocessing again on validation or test data.

## 1. Imports and project paths

In [1]:
from hashlib import sha256
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

print(f"pandas version: {pd.__version__}")
print(f"scikit-learn version: {sklearn.__version__}")
print("Imports completed successfully.")

pandas version: 3.0.5
scikit-learn version: 1.9.0
Imports completed successfully.


In [2]:
WORKING_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    WORKING_DIR.parent
    if WORKING_DIR.name == "notebooks"
    else WORKING_DIR
)

assert (
    PROJECT_ROOT / "docker-compose.yml"
).exists(), "Project root could not be identified."

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

EDA_FINDINGS_PATH = (
    ARTIFACT_DIR
    / "notebook4_eda_findings.json"
)

TRAIN_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "train.parquet"
)

TRAIN_FEATURES_PATH = (
    ARTIFACT_DIR
    / "notebook5_train_features.parquet"
)

PREPROCESSOR_PATH = (
    ARTIFACT_DIR
    / "notebook5_preprocessor.joblib"
)

FEATURE_LIST_PATH = (
    ARTIFACT_DIR
    / "notebook5_feature_list.json"
)

MANIFEST_PATH = (
    ARTIFACT_DIR
    / "notebook5_manifest.json"
)

assert EDA_FINDINGS_PATH.exists(), (
    "Notebook 4 artifact is missing. "
    "Run Notebook 4 first."
)

assert TRAIN_PATH.exists(), (
    "Training split is missing. "
    "Run Notebook 3 first."
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Notebook 4 artifact: {EDA_FINDINGS_PATH}")
print(f"Training split: {TRAIN_PATH}")

Project root: D:\Documents\mlops-olist
Notebook 4 artifact: D:\Documents\mlops-olist\artifacts\notebook4_eda_findings.json
Training split: D:\Documents\mlops-olist\data\processed\train.parquet


## 2. Read and validate Notebook 4 findings

Notebook 5 explicitly consumes the machine-readable artifact produced
by Notebook 4.

This establishes the required notebook-to-notebook artifact contract.


In [3]:
eda_findings = json.loads(
    EDA_FINDINGS_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    eda_findings["producer"]
    == "notebooks/04_train_eda.ipynb"
)

assert (
    eda_findings["training_rows"]
    == 67_529
)

assert (
    eda_findings["validation_opened"]
    is False
)

assert (
    eda_findings["test_opened"]
    is False
)

print(
    f"Notebook 4 training rows: "
    f"{eda_findings['training_rows']:,}"
)

print(
    f"Notebook 4 late rate: "
    f"{eda_findings['late_rate']:.4%}"
)

print(
    f"Notebook 4 leakage columns: "
    f"{len(eda_findings['leakage_columns'])}"
)

print(
    f"Notebook 4 high-cardinality features: "
    f"{len(eda_findings['high_cardinality_features'])}"
)

print("Notebook 4 artifact validation: PASSED")

Notebook 4 training rows: 67,529
Notebook 4 late rate: 7.8337%
Notebook 4 leakage columns: 13
Notebook 4 high-cardinality features: 5
Notebook 4 artifact validation: PASSED


## 3. Load training data only

Feature engineering decisions are fitted using the training split.

Validation and test remain closed until Notebook 6.


In [4]:
train = pd.read_parquet(TRAIN_PATH)

assert train.shape == (67_529, 67)
assert train["order_id"].is_unique
assert train["is_late"].notna().all()

print(f"Training shape: {train.shape}")
print(
    f"Training late rate: "
    f"{train['is_late'].mean():.4%}"
)
print("Validation data opened: NO")
print("Test data opened: NO")

Training shape: (67529, 67)
Training late rate: 7.8337%
Validation data opened: NO
Test data opened: NO


## 4. Define the prediction-time feature contract

The feature set is an explicit whitelist.

Identifiers, target fields, delivery outcomes, and post-delivery
reviews are excluded.

High-cardinality fields such as customer IDs, seller IDs, city names,
and ZIP prefixes are also excluded from the model matrix.

This avoids accidental leakage and excessive sparse dimensionality.


In [5]:
BASE_NUMERIC_FEATURES = [
    "item_count",
    "unique_product_count",
    "unique_seller_count",
    "item_price_total",
    "item_price_mean",
    "item_price_max",
    "freight_value_total",
    "freight_value_mean",
    "unique_product_category_count",
    "product_name_length_mean",
    "product_description_length_mean",
    "product_photos_qty_mean",
    "product_weight_g_total",
    "product_weight_g_mean",
    "product_length_cm_mean",
    "product_height_cm_mean",
    "product_width_cm_mean",
    "primary_category_item_count",
    "unique_seller_city_count",
    "unique_seller_state_count",
    "primary_seller_item_count",
    "payment_record_count",
    "unique_payment_type_count",
    "payment_value_total",
    "payment_value_mean",
    "payment_installments_max",
    "payment_installments_mean",
    "primary_payment_type_value",
    "primary_payment_type_record_count",
    "customer_lat_median",
    "customer_lng_median",
    "customer_geolocation_record_count",
    "primary_seller_lat_median",
    "primary_seller_lng_median",
    "primary_seller_geolocation_record_count",
]

BASE_CATEGORICAL_FEATURES = [
    "customer_record_found",
    "customer_state",
    "primary_seller_state",
    "primary_product_category",
    "primary_payment_type",
]

TIME_SOURCE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date",
    "shipping_limit_date_min",
    "shipping_limit_date_max",
]

FORBIDDEN_COLUMNS = {
    "order_id",
    "customer_id",
    "customer_unique_id",
    "primary_seller_id",
    "order_status",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "delivery_delay_days",
    "post_delivery_review_count",
    "post_delivery_review_score_mean",
    "post_delivery_review_score_min",
    "post_delivery_review_score_max",
    "post_delivery_review_title_count",
    "post_delivery_review_message_count",
    "post_delivery_review_message_length_mean",
    "post_delivery_review_creation_date_min",
    "post_delivery_review_answer_timestamp_max",
    "is_late",
}

raw_predictor_columns = (
    set(BASE_NUMERIC_FEATURES)
    | set(BASE_CATEGORICAL_FEATURES)
    | set(TIME_SOURCE_COLUMNS)
)

forbidden_overlap = (
    raw_predictor_columns
    & FORBIDDEN_COLUMNS
)

assert not forbidden_overlap, (
    f"Forbidden columns selected: "
    f"{sorted(forbidden_overlap)}"
)

missing_raw_columns = sorted(
    raw_predictor_columns
    - set(train.columns)
)

assert not missing_raw_columns, (
    f"Missing source columns: "
    f"{missing_raw_columns}"
)

print(
    f"Base numeric predictors: "
    f"{len(BASE_NUMERIC_FEATURES)}"
)

print(
    f"Base categorical predictors: "
    f"{len(BASE_CATEGORICAL_FEATURES)}"
)

print(
    f"Timestamp sources: "
    f"{len(TIME_SOURCE_COLUMNS)}"
)

print("Forbidden feature overlap: 0")
print("Prediction-time contract: PASSED")

Base numeric predictors: 35
Base categorical predictors: 5
Timestamp sources: 5
Forbidden feature overlap: 0
Prediction-time contract: PASSED


## 5. Deterministic feature engineering

The following prediction-time features implement the implications
recorded by Notebook 4:

- approval delay;
- estimated delivery window;
- shipping-limit durations;
- purchase calendar features;
- approximate customer?seller distance;
- freight-to-price ratio;
- whether customer and seller are in the same state.

These are deterministic transformations.

They do not learn parameters from validation or test data.


In [6]:
def haversine_km(
    lat1,
    lon1,
    lat2,
    lon2,
):
    lat1 = np.radians(
        pd.to_numeric(
            lat1,
            errors="coerce",
        )
    )

    lon1 = np.radians(
        pd.to_numeric(
            lon1,
            errors="coerce",
        )
    )

    lat2 = np.radians(
        pd.to_numeric(
            lat2,
            errors="coerce",
        )
    )

    lon2 = np.radians(
        pd.to_numeric(
            lon2,
            errors="coerce",
        )
    )

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return (
        6371.0088
        * 2
        * np.arcsin(
            np.sqrt(a)
        )
    )


def build_prediction_features(frame):
    X = frame[
        BASE_NUMERIC_FEATURES
        + BASE_CATEGORICAL_FEATURES
    ].copy()

    for column in BASE_NUMERIC_FEATURES:
        X[column] = pd.to_numeric(
            X[column],
            errors="coerce",
        )

    for column in BASE_CATEGORICAL_FEATURES:
        X[column] = X[column].astype(
            "object"
        )

        X.loc[
            X[column].isna(),
            column,
        ] = np.nan

    purchase = pd.to_datetime(
        frame["order_purchase_timestamp"],
        errors="coerce",
    )

    approved = pd.to_datetime(
        frame["order_approved_at"],
        errors="coerce",
    )

    estimated = pd.to_datetime(
        frame[
            "order_estimated_delivery_date"
        ],
        errors="coerce",
    )

    shipping_min = pd.to_datetime(
        frame["shipping_limit_date_min"],
        errors="coerce",
    )

    shipping_max = pd.to_datetime(
        frame["shipping_limit_date_max"],
        errors="coerce",
    )

    X["approval_delay_hours"] = (
        approved - purchase
    ).dt.total_seconds() / 3600

    X[
        "estimated_delivery_window_days"
    ] = (
        estimated - purchase
    ).dt.total_seconds() / 86400

    X["shipping_limit_min_days"] = (
        shipping_min - purchase
    ).dt.total_seconds() / 86400

    X["shipping_limit_max_days"] = (
        shipping_max - purchase
    ).dt.total_seconds() / 86400

    X["shipping_limit_span_days"] = (
        shipping_max - shipping_min
    ).dt.total_seconds() / 86400

    X["purchase_month"] = (
        purchase.dt.month.astype(
            "float64"
        )
    )

    X["purchase_day_of_week"] = (
        purchase.dt.dayofweek.astype(
            "float64"
        )
    )

    X["purchase_hour"] = (
        purchase.dt.hour.astype(
            "float64"
        )
    )

    X[
        "customer_seller_distance_km"
    ] = haversine_km(
        frame["customer_lat_median"],
        frame["customer_lng_median"],
        frame[
            "primary_seller_lat_median"
        ],
        frame[
            "primary_seller_lng_median"
        ],
    )

    X["freight_to_price_ratio"] = (
        pd.to_numeric(
            frame["freight_value_total"],
            errors="coerce",
        )
        /
        pd.to_numeric(
            frame["item_price_total"],
            errors="coerce",
        ).replace(
            0,
            np.nan,
        )
    )

    same_state = (
        frame["customer_state"]
        .eq(
            frame[
                "primary_seller_state"
            ]
        )
    )

    missing_state = (
        frame[
            [
                "customer_state",
                "primary_seller_state",
            ]
        ]
        .isna()
        .any(axis=1)
    )

    X[
        "customer_seller_same_state"
    ] = (
        same_state
        .mask(
            missing_state,
            np.nan,
        )
        .astype(float)
    )

    return X


X_train_raw = build_prediction_features(
    train
)

y_train = train[
    "is_late"
].astype("int8")

print(
    f"Rows after feature engineering: "
    f"{len(X_train_raw):,}"
)

print(
    f"Predictor columns before encoding: "
    f"{X_train_raw.shape[1]}"
)

print(
    "Rows removed by feature engineering: 0"
)

Rows after feature engineering: 67,529
Predictor columns before encoding: 51
Rows removed by feature engineering: 0


## 6. Fit preprocessing on training data only

Numerical preprocessing:

- median imputation;
- missing-value indicators;
- standardization.

Categorical preprocessing:

- most-frequent imputation;
- one-hot encoding;
- infrequent categories grouped using the training data.

The fitted preprocessing object is an artifact.

Notebook 6 must **load this object** and use `.transform(...)`.
It must not call `.fit(...)` or `.fit_transform(...)` on validation
or test data.


In [7]:
numeric_features = [
    column
    for column
    in X_train_raw.columns
    if column
    not in BASE_CATEGORICAL_FEATURES
]

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
            ),
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                min_frequency=20,
                sparse_output=False,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            BASE_CATEGORICAL_FEATURES,
        ),
    ],
    verbose_feature_names_out=False,
)

X_train_transformed = (
    preprocessor.fit_transform(
        X_train_raw
    )
)

feature_names = (
    preprocessor
    .get_feature_names_out()
    .tolist()
)

assert (
    X_train_transformed.shape[0]
    == len(train)
)

assert (
    X_train_transformed.shape[1]
    == len(feature_names)
)

assert len(feature_names) == len(
    set(feature_names)
)

print(
    f"Training rows fitted: "
    f"{len(train):,}"
)

print(
    f"Model-ready features: "
    f"{len(feature_names)}"
)

print(
    "Preprocessor fitted on: TRAIN ONLY"
)

print(
    "Validation used during fitting: NO"
)

print(
    "Test used during fitting: NO"
)

Training rows fitted: 67,529
Model-ready features: 184
Preprocessor fitted on: TRAIN ONLY
Validation used during fitting: NO
Test used during fitting: NO


## 7. Inspect the ordered final feature list

The feature order is part of the model contract.

Notebook 6 must use this exact order after applying the saved
preprocessor.


In [8]:
feature_list_table = pd.DataFrame(
    {
        "position": range(
            len(feature_names)
        ),
        "feature": feature_names,
    }
)

display(
    feature_list_table.head(50)
)

print(
    f"Ordered features recorded: "
    f"{len(feature_names)}"
)

,position,feature
0,0,item_count
1,1,unique_product_count
2,2,unique_seller_count
3,3,item_price_total
4,4,item_price_mean
5,5,item_price_max
6,6,freight_value_total
7,7,freight_value_mean
8,8,unique_product_category_count
9,9,product_name_length_mean


Ordered features recorded: 184


## 8. Build the final training feature table

The transformed training matrix is stored with its target.

No identifier or leakage column is included in the model feature
matrix.


In [9]:
X_train_final = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=train.index,
).astype("float32")

train_feature_table = (
    X_train_final.copy()
)

train_feature_table[
    "is_late"
] = y_train.to_numpy()

assert (
    train_feature_table.shape[0]
    == 67_529
)

assert (
    train_feature_table[
        "is_late"
    ].notna().all()
)

assert (
    set(
        train_feature_table[
            "is_late"
        ].unique()
    )
    == {0, 1}
)

assert not (
    set(feature_names)
    & FORBIDDEN_COLUMNS
)

print(
    f"Final training feature table: "
    f"{train_feature_table.shape}"
)

print(
    f"Target late rate: "
    f"{train_feature_table['is_late'].mean():.4%}"
)

print(
    "Leakage columns in feature matrix: 0"
)

Final training feature table: (67529, 185)
Target late rate: 7.8337%
Leakage columns in feature matrix: 0


## 9. Save Notebook 5 artifacts

Notebook 5 saves:

1. the final training feature table;
2. the fitted preprocessing object;
3. the ordered feature list;
4. a manifest containing hashes and fitting provenance.

All outputs are reload-validated immediately.


In [10]:
ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

train_feature_table.to_parquet(
    TRAIN_FEATURES_PATH,
    index=False,
)

joblib.dump(
    preprocessor,
    PREPROCESSOR_PATH,
)

FEATURE_LIST_PATH.write_text(
    json.dumps(
        feature_names,
        indent=2,
    ),
    encoding="utf-8",
)


def file_sha256(path):
    return (
        sha256(
            path.read_bytes()
        )
        .hexdigest()
        .upper()
    )


reloaded_train_features = (
    pd.read_parquet(
        TRAIN_FEATURES_PATH
    )
)

reloaded_preprocessor = (
    joblib.load(
        PREPROCESSOR_PATH
    )
)

reloaded_feature_names = (
    json.loads(
        FEATURE_LIST_PATH.read_text(
            encoding="utf-8"
        )
    )
)

pd.testing.assert_frame_equal(
    train_feature_table,
    reloaded_train_features,
)

assert (
    reloaded_feature_names
    == feature_names
)

assert (
    reloaded_preprocessor
    .get_feature_names_out()
    .tolist()
    == feature_names
)

manifest = {
    "producer": (
        "notebooks/"
        "05_feature_engineering.ipynb"
    ),
    "previous_step_artifact": (
        "artifacts/"
        "notebook4_eda_findings.json"
    ),
    "previous_step_sha256": (
        file_sha256(
            EDA_FINDINGS_PATH
        )
    ),
    "training_input": (
        "data/processed/train.parquet"
    ),
    "training_input_sha256": (
        file_sha256(
            TRAIN_PATH
        )
    ),
    "fitted_on_split": "train",
    "training_rows": int(
        len(train)
    ),
    "raw_predictor_count": int(
        X_train_raw.shape[1]
    ),
    "final_feature_count": int(
        len(feature_names)
    ),
    "target": "is_late",
    "artifacts": {
        "train_feature_table": {
            "path": (
                "artifacts/"
                "notebook5_train_features.parquet"
            ),
            "sha256": file_sha256(
                TRAIN_FEATURES_PATH
            ),
        },
        "fitted_preprocessor": {
            "path": (
                "artifacts/"
                "notebook5_preprocessor.joblib"
            ),
            "sha256": file_sha256(
                PREPROCESSOR_PATH
            ),
        },
        "feature_list": {
            "path": (
                "artifacts/"
                "notebook5_feature_list.json"
            ),
            "sha256": file_sha256(
                FEATURE_LIST_PATH
            ),
        },
    },
    "validation_opened": False,
    "test_opened": False,
    "model_trained": False,
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)

reloaded_manifest = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

assert (
    reloaded_manifest
    == manifest
)

print(
    "All Notebook 5 artifacts "
    "reload validation: PASSED"
)

print(
    f"Train feature SHA-256: "
    f"{file_sha256(TRAIN_FEATURES_PATH)}"
)

print(
    f"Preprocessor SHA-256: "
    f"{file_sha256(PREPROCESSOR_PATH)}"
)

print(
    f"Feature list SHA-256: "
    f"{file_sha256(FEATURE_LIST_PATH)}"
)

print(
    f"Manifest SHA-256: "
    f"{file_sha256(MANIFEST_PATH)}"
)

All Notebook 5 artifacts reload validation: PASSED
Train feature SHA-256: 50AD50B397EC0171D8CED14279D4671F38CEDC96CD7CD29DA3E1777F5B33E945
Preprocessor SHA-256: 33B34D3781BA235DD5C2B1570F156E4A8C4F04FE78DE56B629E59A017010DA31
Feature list SHA-256: 04227488CAA8BB1A2AC407FFCCDDF06154396EC8CC1C376CDC93077242140434
Manifest SHA-256: 07A860F168483920D5F131E2D2AB224B5EDAB8CA898CC6DABD2CD112C1660E1A


## 10. Notebook 5 completion contract

Notebook 5 is complete when:

- Notebook 4's artifact is read successfully;
- only training data is used;
- prediction-time feature engineering is deterministic;
- identifiers and leakage fields are excluded;
- the preprocessor is fitted on training data only;
- the final transformed training feature table is saved;
- the fitted preprocessor is saved;
- the ordered feature list is saved;
- all artifacts reload successfully;
- validation remains unopened;
- test remains unopened;
- no model is trained.

Notebook 6 will load these exact fitted artifacts to perform baseline,
training, validation tuning, and the final one-time test evaluation.


In [11]:
print("NOTEBOOK 5 FINAL RESULT")
print(
    f"Training rows: "
    f"{len(train):,}"
)
print(
    f"Raw predictor columns: "
    f"{X_train_raw.shape[1]}"
)
print(
    f"Final encoded features: "
    f"{len(feature_names)}"
)
print(
    f"Final feature-table shape: "
    f"{train_feature_table.shape}"
)
print(
    "Previous Notebook 4 artifact read: YES"
)
print(
    "Preprocessor fitted on train only: YES"
)
print(
    "Final training feature table saved: YES"
)
print(
    "Fitted preprocessor saved: YES"
)
print(
    "Ordered feature list saved: YES"
)
print(
    "Artifact manifest saved: YES"
)
print(
    "Artifact reload validation: PASSED"
)
print(
    "Validation data opened: NO"
)
print(
    "Test data opened: NO"
)
print(
    "Model trained: NO"
)
print(
    "Ready for corrected Notebook 6: YES"
)

NOTEBOOK 5 FINAL RESULT
Training rows: 67,529
Raw predictor columns: 51
Final encoded features: 184
Final feature-table shape: (67529, 185)
Previous Notebook 4 artifact read: YES
Preprocessor fitted on train only: YES
Final training feature table saved: YES
Fitted preprocessor saved: YES
Ordered feature list saved: YES
Artifact manifest saved: YES
Artifact reload validation: PASSED
Validation data opened: NO
Test data opened: NO
Model trained: NO
Ready for corrected Notebook 6: YES
